# Задание 10*. Анализ тональности с помощью Hugging Face

Задача: бинарная классификация тональности коротких отзывов о фильмах (Negative / Positive).

- Датасет: `cornell-movie-review-data/rotten_tomatoes` (train 8530, validation 1066, test 1066).
- Модель: `distilbert-base-uncased`, дообучение через `Trainer`.
- Метрики: Accuracy и F1 (класс Positive) на validation и test.

In [8]:
import numpy as np
import torch
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)
MODEL_NAME = "distilbert-base-uncased"
SAVE_DIR = "task10_model"  # сюда сохраняется итоговая модель и из нее же загружается
# True - обучить модель заново (около 30 минут на CPU) и сохранить в SAVE_DIR
# False - загрузить уже обученную модель из SAVE_DIR без обучения
TRAIN = False

In [9]:
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print(dataset)
print("Пример:", dataset["train"][0])
for split in ["train", "validation", "test"]:
    print(split, "метки:", dict(Counter(dataset[split]["label"])))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
Пример: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
train метки: {1: 4265, 0: 4265}
validation метки: {1: 533, 0: 533}
test метки: {1: 533, 0: 533}


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)
print(tokenizer.convert_ids_to_tokens(tokenized["train"][0]["input_ids"]))

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1066
    })
})
['[CLS]', 'the', 'rock', 'is', 'destined', 'to', 'be', 'the', '21st', 'century', "'", 's', 'new', '"', 'conan', '"', 'and', 'that', 'he', "'", 's', 'going', 'to', 'make', 'a', 'splash', 'even', 'greater', 'than', 'arnold', 'schwarz', '##ene', '##gger', ',', 'jean', '-', 'cl', '##aud', 'van', 'dam', '##me', 'or', 'steven', 'sega', '##l', '.', '[SEP]']


In [11]:
id2label = {0: "Negative", 1: "Positive"}
label2id = {v: k for k, v in id2label.items()}

if TRAIN:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
    )
else:
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
print("Параметров модели:", sum(p.numel() for p in model.parameters()))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 802.54it/s]

Параметров модели: 66955010


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

In [13]:
args = TrainingArguments(
    output_dir="outputs_task10",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [14]:
if TRAIN:
    train_result = trainer.train()
    print(train_result)
    # Сохраняем лучшую модель (по F1 на validation) и токенизатор
    trainer.save_model(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)
    print("Модель сохранена в", SAVE_DIR)
else:
    print("Обучение пропущено, модель загружена из", SAVE_DIR)

Обучение пропущено, модель загружена из task10_model


In [15]:
# Метрики на validation (по нему выбрана лучшая эпоха)
val_metrics = trainer.evaluate(tokenized["validation"])
print("Validation: accuracy = {:.4f}, F1 = {:.4f}, loss = {:.4f}".format(
    val_metrics["eval_accuracy"], val_metrics["eval_f1"], val_metrics["eval_loss"]))

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.365458,0,0.854597,0.853636


Validation: accuracy = 0.8546, F1 = 0.8536, loss = 0.3655


In [16]:
# Метрики на test (не использовался ни для обучения, ни для выбора модели)
test_output = trainer.predict(tokenized["test"])
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

print("Test: accuracy = {:.4f}, F1 = {:.4f}".format(
    accuracy_score(test_labels, test_preds), f1_score(test_labels, test_preds)))
print("Матрица ошибок (строки - истина, столбцы - предсказание, порядок Negative, Positive):")
print(confusion_matrix(test_labels, test_preds))

c:\Users\eliassen\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test: accuracy = 0.8452, F1 = 0.8436
Матрица ошибок (строки - истина, столбцы - предсказание, порядок Negative, Positive):
[[456  77]
 [ 88 445]]


In [17]:
# Произвольные тексты, не входящие в датасет. Ожидаемая метка задана вручную (1 - Positive, 0 - Negative)
custom_texts = [
    ("An absolutely brilliant film with a superb cast and a heartfelt ending.", 1),
    ("I was bored to tears and checked my watch every ten minutes.", 0),
    ("This is one of the best comedies I have seen in years.", 1),
    ("The script is a mess, and the acting is painfully bad.", 0),
    ("Not a single moment of this movie felt genuine.", 0),
    ("It is not bad at all, I actually enjoyed it.", 1),
    ("Oh great, another sequel nobody asked for. Just what we needed.", 0),
    ("The director clearly knows what he is doing, and every scene looks wonderful.", 1),
]

model = trainer.model
model.eval()
enc = tokenizer([t for t, _ in custom_texts], truncation=True, max_length=64,
                padding=True, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1).numpy()

correct = 0
for (text, expected), p in zip(custom_texts, probs):
    pred = int(p.argmax())
    correct += pred == expected
    mark = "+" if pred == expected else "-"
    print(f"{mark} {id2label[pred]:<9} P(Positive)={p[1]:.3f}  ожидаемо: {id2label[expected]:<9} | {text}")
print("Верно:", correct, "из", len(custom_texts))

+ Positive  P(Positive)=0.993  ожидаемо: Positive  | An absolutely brilliant film with a superb cast and a heartfelt ending.
+ Negative  P(Positive)=0.056  ожидаемо: Negative  | I was bored to tears and checked my watch every ten minutes.
+ Positive  P(Positive)=0.982  ожидаемо: Positive  | This is one of the best comedies I have seen in years.
+ Negative  P(Positive)=0.013  ожидаемо: Negative  | The script is a mess, and the acting is painfully bad.
+ Negative  P(Positive)=0.049  ожидаемо: Negative  | Not a single moment of this movie felt genuine.
+ Positive  P(Positive)=0.951  ожидаемо: Positive  | It is not bad at all, I actually enjoyed it.
- Positive  P(Positive)=0.705  ожидаемо: Negative  | Oh great, another sequel nobody asked for. Just what we needed.
+ Positive  P(Positive)=0.986  ожидаемо: Positive  | The director clearly knows what he is doing, and every scene looks wonderful.
Верно: 7 из 8
